In [ ]:
# !pip install fastmcp strands-agents


  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.0.1
    Uninstalling python-dotenv-1.0.1:
      Successfully uninstalled python-dotenv-1.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23/23 [fastmcp]2/23 [fastmcp]ma]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
graphrag-toolkit-lexical-graph 3.9.2 requires python-dotenv==1.0.1, but you have python-dotenv 1.1.1 which is incompatible.


## Create an MCP Server

In [2]:
import logging

from graphrag_toolkit.lexical_graph import set_advanced_logging_config

set_advanced_logging_config(
    logging_level=logging.DEBUG,
    included_modules={
        logging.DEBUG: [
            'graphrag_toolkit.lexical_graph.protocols', 
            'graphrag_toolkit.lexical_graph.retrieval.summary'
        ],
        logging.INFO: '*',
    },
    excluded_modules={
        logging.DEBUG: ['opensearch', 'boto', 'urllib'],
        logging.INFO: ['opensearch', 'boto', 'urllib', 'mcp', 'httpx'],
    }
)

In [3]:
%reload_ext dotenv
%dotenv

import os

from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory, VectorStoreFactory
from graphrag_toolkit.lexical_graph.protocols import create_mcp_server

graph_store = GraphStoreFactory.for_graph_store(os.environ['GRAPH_STORE'])
vector_store = VectorStoreFactory.for_vector_store(os.environ['VECTOR_STORE'])

mcp_server = create_mcp_server(graph_store, vector_store)

print('Server initialized')

2025-07-07 19:04:56:DEBUG:g.l.r.s.graph_summary:No cached summary found for 'default_'
2025-07-07 19:04:56:DEBUG:g.l.r.s.graph_summary:Creating new summary for 'default_'
2025-07-07 19:05:01:DEBUG:g.l.r.s.graph_summary:Caching summary for 'default_'
Server initialized


### Start the server

In [ ]:
import threading

def run_server():
    mcp_server.run(transport='streamable-http', log_level='warning')
    
thread = threading.Thread(target=run_server)
thread.start()



╭─ FastMCP 2.0 ────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│         _ __ ___ ______           __  __  _____________       ____           │
│     ____                                                                     │
│        _ __ ___ / ____/___ ______/ /_/  |/  / ____/ __ \     |___ \  /       │
│     __ \                                                                     │
│       _ __ ___ / /_  / __ `/ ___/ __/ /|_/ / /   / /_/ /     ___/ / / /      │
│     / /                                                                      │
│      _ __ ___ / __/ / /_/ (__  ) /_/ /  / / /___/ ____/     /  __/_/ /_/     │
│     /                                                                        │
│     _ __ ___ /_/    \__,_/____/\__/_/  /_/\____/_/                           │
│     /_____(_)____/      

[07/07/25 19:05:25] INFO     Starting MCP server 'LexicalGraphServer' with transport                 ]8;id=551673;file:///opt/homebrew/Caskroom/miniforge/base/envs/graphrag-py310/lib/python3.10/site-packages/fastmcp/server/server.py\server.py]8;;\:]8;id=761103;file:///opt/homebrew/Caskroom/miniforge/base/envs/graphrag-py310/lib/python3.10/site-packages/fastmcp/server/server.py#1429\1429]8;;\
                             'streamable-http' on http://127.0.0.1:8000/mcp/                                       

2025-07-07 19:07:29:DEBUG:g.l.p.mcp_server:[default_]: What are SLMs or what does SLM mean in the context of language models [5 results, 6727 millis]
2025-07-07 19:07:39:DEBUG:g.l.p.mcp_server:[default_]: Statistical Language Models (SLMs) in natural language processing [5 results, 7365 millis]
2025-07-07 19:11:08:DEBUG:g.l.p.mcp_server:[default_]: What are SLMs or what does SLM mean in the context of language models? [5 results, 5046 millis]
2025-07-07 19:11:19:DEBUG:g.l.p.mcp_server:[default_]: What are Statistical Language Models (SLMs) in natural language processing? [5 results, 7803 millis]
2025-07-07 19:16:14:ERROR:m.s.streamable_http:Error in standalone SSE writer: 
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniforge/base/envs/graphrag-py310/lib/python3.10/site-packages/mcp/server/streamable_http.py", line 572, in standalone_sse_writer
    async for event_message in standalone_stream_reader:
  File "/opt/homebrew/Caskroom/miniforge/base/envs/graphrag-py31

## Create an MCP client and AI agent

In [5]:
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient

def create_streamable_http_transport():
    return streamablehttp_client('http://localhost:8000/mcp/')

mcp_client = MCPClient(create_streamable_http_transport)

In [6]:
from strands import Agent

with mcp_client:
    
    tools = mcp_client.list_tools_sync()
    
    for tool in tools:
        print(f"{tool.tool_spec['name']}: {tool.tool_spec['description']}")
        print('\n-------------------------------------\n')

default_: Domain: The knowledge base covers the domain of large language models (LLMs), their architectures, variants, and applications in natural language processing.

Scope: The knowledge base includes information on various LLM architectures, such as BERT, RoBERTa, ALBERT, DeBERTa, and Transformers. It also covers the concepts of statistical language modeling, pre-trained language models, and their use in tasks like natural language generation and understanding. The knowledge base includes information on the researchers and organizations involved in the development of these models.

Uses: This knowledge base can be used to:
1. Understand the different types of LLMs, their architectures, and their capabilities.
2. Explore the historical development and evolution of LLMs.
3. Identify the key researchers and organizations working on LLM development.
4. Investigate the applications and use cases of LLMs in natural language processing.
5. Analyze the similarities and differences between 

## Create an agent and ask a question
We can now create a Strands AI Agent, and ask a question. The agent will choose the most appropriate tools for answering the question.

In [12]:
with mcp_client:

    tools = mcp_client.list_tools_sync()
    print("--- Available Tools ---")
    # Iterate through the list and print the name and description for each
    for tool in tools:
        print(f"Name: {tool.tool_spec['name']}")
        print(f"Description: {tool.tool_spec['description']}\n")

--- Available Tools ---
Name: default_
Description: Domain: The knowledge base covers the domain of large language models (LLMs), their architectures, variants, and applications in natural language processing.

Scope: The knowledge base includes information on various LLM architectures, such as BERT, RoBERTa, ALBERT, DeBERTa, and Transformers. It also covers the concepts of statistical language modeling, pre-trained language models, and their use in tasks like natural language generation and understanding. The knowledge base includes information on the researchers and organizations involved in the development of these models.

Uses: This knowledge base can be used to:
1. Understand the different types of LLMs, their architectures, and their capabilities.
2. Explore the historical development and evolution of LLMs.
3. Identify the key researchers and organizations working on LLM development.
4. Investigate the applications and use cases of LLMs in natural language processing.
5. Analyze

In [10]:
with mcp_client:

    tools = mcp_client.list_tools_sync()
    print("Tools: ", tools)
    agent = Agent(tools=tools)
    
    response = agent("What are SLMs mean?")

Tools:  [<strands.tools.mcp.mcp_agent_tool.MCPAgentTool object at 0x331433af0>, <strands.tools.mcp.mcp_agent_tool.MCPAgentTool object at 0x33e111210>]
I'll help you understand what "SLMs" means in the context of language models. Let me search for information about this term.
Tool #1: default_
I apologize for the error. Let me try a more specific query about SLMs in the context of language models.
Tool #2: default_
I apologize for the technical difficulties. Let me try a different approach to answer your question about what SLMs mean.

Based on my knowledge, SLMs typically stands for "Statistical Language Models" in the context of natural language processing and machine learning.

Statistical Language Models (SLMs) are probabilistic models that assign probabilities to sequences of words or tokens. They are fundamental to many natural language processing tasks including:

1. Speech recognition
2. Machine translation
3. Information retrieval
4. Text generation
5. Spell checking and correc

In [11]:
# Inspect the complete conversation flow
def inspect_message_flow(messages):
    print("=== DETAILED MESSAGE FLOW ===")
    
    for i, message in enumerate(messages):
        print(f"\n--- Message {i+1} ---")
        print(f"Role: {message['role']}")
        
        for j, content in enumerate(message['content']):
            print(f"  Content {j+1}:")
            
            if 'text' in content:
                text = content['text']
                # Truncate long text for readability
                if len(text) > 200:
                    text = text[:200] + "..."
                print(f"    Text: {text}")
            
            elif 'toolUse' in content:
                tool_use = content['toolUse']
                print(f"    Tool Use: {tool_use['name']}")
                print(f"    Input: {tool_use['input']}")
                print(f"    ID: {tool_use['toolUseId']}")
            
            elif 'toolResult' in content:
                tool_result = content['toolResult']
                print(f"    Tool Result: {tool_result['status']}")
                print(f"    ID: {tool_result['toolUseId']}")
                # Don't print full content as it's very long
                print(f"    Content: [Raw KB Response - {len(str(tool_result['content']))} chars]")

# Run the inspection
inspect_message_flow(agent.messages)

=== DETAILED MESSAGE FLOW ===

--- Message 1 ---
Role: user
  Content 1:
    Text: What are SLMs mean?

--- Message 2 ---
Role: assistant
  Content 1:
    Text: I'll help you understand what "SLMs" means in the context of language models. Let me search for information about this term.
  Content 2:
    Tool Use: default_
    Input: {'query': 'What are SLMs or what does SLM mean in the context of language models?'}
    ID: tooluse_XK9soPIZTneKoeUrofcZng

--- Message 3 ---
Role: user
  Content 1:
    Tool Result: error
    ID: tooluse_XK9soPIZTneKoeUrofcZng
    Content: [Raw KB Response - 1164 chars]

--- Message 4 ---
Role: assistant
  Content 1:
    Text: I apologize for the error. Let me try a more specific query about SLMs in the context of language models.
  Content 2:
    Tool Use: default_
    Input: {'query': 'What are Statistical Language Models (SLMs) in natural language processing?'}
    ID: tooluse_OkbD1BZUQvGHBxuhC72uyw

--- Message 5 ---
Role: user
  Content 1:
    Tool Resu

In [23]:
# import json

# # The user's question
# query_text = "What are SLMs mean?"

# # The name of the tool we want to call directly.
# # We know from the previous step the main RAG tool is named 'default_'.
# tool_name = "default_"

# # The input for the tool must be a dictionary. For the 'default_' tool,
# # the required argument is 'query'.
# tool_input = {"query": query_text}

# print(f"--> Directly calling tool: '{tool_name}' with query: '{query_text}'")

# # Use a 'with' block to ensure the client connection is handled correctly.
# with mcp_client:
#     # This is the direct tool invocation.
#     # We provide the tool's name and the structured input it expects.
#     tool_result = mcp_client.call_tool_sync(
#         name=tool_name, 
#         arguments=tool_input
#     )

# # The 'tool_result' is the raw output from the knowledge base.
# # It's a list of dictionaries, where each one is a retrieved source node.
# # We can loop through and print them to see the data.
# print("\n<-- Direct Tool Result ---")
# for i, result_item in enumerate(tool_result):
#     print(f"\n[Retrieved Block {i+1}]")
#     # Use json.dumps for pretty-printing the complex dictionary
#     print(json.dumps(result_item, indent=2))


In [27]:
import json

query_text = "What are some of the earliest language models?"
tool_name   = "default_"          # the RAG tool
tool_input  = {"query": query_text}

with mcp_client:
    tool_result = mcp_client.call_tool_sync(name=tool_name, arguments=tool_input)

    print("Raw tool result:")
    print(json.dumps(tool_result, indent=2))
    
    if tool_result.get('status') == 'error':
        print(f"\nError occurred: {tool_result['content'][0]['text']}")
    else:
        print(f"\nTool result: {tool_result['content'][0]['text']}")


TypeError: MCPClient.call_tool_sync() missing 1 required positional argument: 'tool_use_id'